# Warp shuffles and reductions: softmax and RMSNorm kernels

How threads in a warp exchange registers directly with shuffle instructions, how to build a warp and a block reduction from them, and how to use those to write the two reductions every transformer runs constantly: RMSNorm and softmax, including the online softmax that makes FlashAttention possible.

This notebook holds the code from the lesson **[Warp shuffles and reductions: softmax and RMSNorm kernels](https://inferenceacademy.com/learn/kern-cuda/warp-reductions-softmax-and-rmsnorm)** on Inference Academy, in the same order. Read the lesson for the explanations, diagrams and exercises, and run the cells here as you go.

**Before you start:** choose *Runtime → Change runtime type → T4 GPU*, then run the first cell to check the GPU.

In [ ]:
!nvidia-smi

## Warp shuffles

Snippet from the lesson (add it to your program; not run on its own):

```cpp
float v = ...;                                  // each lane has a value
float w = __shfl_xor_sync(0xffffffff, v, 16);   // read v from lane (my_lane XOR 16)
```

Snippet from the lesson (add it to your program; not run on its own):

```cpp
__device__ float warp_sum(float v) {
  for (int o = 16; o > 0; o >>= 1) v += __shfl_xor_sync(0xffffffff, v, o);
  return v;
}
```

## Block reductions

Snippet from the lesson (add it to your program; not run on its own):

```cpp
// Sum over all threads of a block. blockDim.x must be a multiple of 32 (up to 1,024).
__device__ float block_sum(float v) {
  __shared__ float part[32];
  int lane = threadIdx.x % 32, warp = threadIdx.x / 32;
  v = warp_sum(v);
  if (lane == 0) part[warp] = v;
  __syncthreads();
  v = (threadIdx.x < blockDim.x / 32) ? part[lane] : 0.f;
  if (warp == 0) v = warp_sum(v);
  if (threadIdx.x == 0) part[0] = v;
  __syncthreads();
  return part[0];             // every thread gets the total
}
```

## RMSNorm

Snippet from the lesson (add it to your program; not run on its own):

```cpp
#include <cuda_fp16.h>

// x, y: [rows, d] FP16, w: [d] FP16. Launch: rmsnorm<<<rows, 256>>>(x, w, y, d, 1e-6f)
__global__ void rmsnorm(const half* x, const half* w, half* y, int d, float eps) {
  const half* xr = x + (size_t)blockIdx.x * d;
  half* yr = y + (size_t)blockIdx.x * d;
  float ss = 0.f;
  for (int i = threadIdx.x; i < d; i += blockDim.x) { float v = __half2float(xr[i]); ss += v * v; }
  float inv = rsqrtf(block_sum(ss) / d + eps);
  for (int i = threadIdx.x; i < d; i += blockDim.x)
    yr[i] = __float2half(__half2float(xr[i]) * inv * __half2float(w[i]));
}
```

### What to expect

In [ ]:
import torch, triton
x = torch.randn(4096, 4096, device="cuda", dtype=torch.half)
w = torch.randn(4096, device="cuda", dtype=torch.half)
norm = lambda: torch.nn.functional.rms_norm(x, (4096,), w, 1e-6)   # PyTorch 2.4 or later
print(triton.testing.do_bench(norm), "ms")

## Softmax, and the online trick

Snippet from the lesson (add it to your program; not run on its own):

```cpp
#include <cfloat>
struct MD { float m, d; };
__device__ MD md_merge(MD a, MD b) {
  float m = fmaxf(a.m, b.m);
  return { m, a.d * __expf(a.m - m) + b.d * __expf(b.m - m) };
}
__device__ MD warp_md(MD v) {
  for (int o = 16; o > 0; o >>= 1)
    v = md_merge(v, { __shfl_xor_sync(0xffffffff, v.m, o), __shfl_xor_sync(0xffffffff, v.d, o) });
  return v;
}

// One block per row of x: [rows, n] FP32.
__global__ void softmax_online(const float* x, float* y, int n) {
  const float* xr = x + (size_t)blockIdx.x * n;
  float* yr = y + (size_t)blockIdx.x * n;
  MD t = { -FLT_MAX, 0.f };                        // -FLT_MAX, not -INFINITY: avoids inf - inf
  for (int i = threadIdx.x; i < n; i += blockDim.x) t = md_merge(t, { xr[i], 1.f });

  __shared__ MD part[32];
  int lane = threadIdx.x % 32, warp = threadIdx.x / 32;
  t = warp_md(t);
  if (lane == 0) part[warp] = t;
  __syncthreads();
  t = (threadIdx.x < blockDim.x / 32) ? part[lane] : MD{ -FLT_MAX, 0.f };
  if (warp == 0) t = warp_md(t);
  if (threadIdx.x == 0) part[0] = t;
  __syncthreads();
  MD r = part[0];

  for (int i = threadIdx.x; i < n; i += blockDim.x) yr[i] = __expf(xr[i] - r.m) / r.d;
}
```

### Why online softmax matters so much

## Where else reductions appear

## What you can now do

## Common mistakes

## Exercise

## Going deeper

---
Code from [Inference Academy](https://inferenceacademy.com), MIT licensed. Lesson: https://inferenceacademy.com/learn/kern-cuda/warp-reductions-softmax-and-rmsnorm